In [1]:
from helper_functions import *
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models
import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error

2026-06-08 16:32:11.090246: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
all_wind_data_1 = pd.read_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject/All_wind_data_2.csv')
flights = pd.read_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject/y_data_rms.csv')


In [3]:
df = all_wind_data_1.merge(flights[['source_file', 'bottom_rms']], on='source_file').reset_index(drop=True)
df = df.dropna(subset=['bottom_rms'])

X = df.drop(columns=['bottom_rms'])
y = df['bottom_rms']

def add_uv(df, prefix):
    speed = df[f"{prefix}.VectorMeanWindSpeed"]
    direction = df[f"{prefix}.VectorMeanWindDirection"]

    df[f"{prefix}_u"] = speed * np.cos(direction)
    df[f"{prefix}_v"] = speed * np.sin(direction)

    return df

all_wind_data = add_uv(X, 'W04')
all_wind_data = add_uv(X, 'W22')
all_wind_data.drop(columns=['W04.VectorMeanWindSpeed', 'W04.VectorMeanWindDirection', 'W22.VectorMeanWindSpeed', 'W22.VectorMeanWindDirection'], inplace=True)

print(all_wind_data.shape)
print(flights.shape)

(335714, 31)
(2827, 11)


In [4]:
feature_cols = all_wind_data.columns.tolist()[8:]

feature_cols.remove('W04.OkPct')
feature_cols.remove('W22.OkPct')
feature_cols.remove('source_file')

flights_lookup = (
    flights
    .set_index("source_file")["bottom_rms"])
X_list = []
y_list = []

for source_file, group in all_wind_data.groupby("source_file"):

    if source_file not in flights_lookup.index:
        continue

    seq = group[feature_cols].values

    X_list.append(seq)
    y_list.append(flights_lookup.loc[source_file])

lengths = [len(x) for x in X_list]

min_len = 119

X = np.stack([
    seq[:min_len]
    for seq in X_list
])

print(X.shape)
y = np.array(y_list)
print(y.shape)

(2793, 119, 20)
(2793,)


In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False)

X_train, X_test, y_train, y_test = train_test_split(
    X_train, y_train,
    test_size=0.5,
    shuffle=False)

n_samples, n_steps, n_features = X_train.shape
n_val = X_val.shape[0]
n_test = X_test.shape[0]

scaler = StandardScaler()
y_scaler = StandardScaler()


X_train_reshaped = X_train.reshape(-1, n_features)
X_val_reshaped = X_val.reshape(-1, n_features)
X_test_reshaped = X_test.reshape(-1, n_features)

X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_val_scaled = scaler.transform(X_val_reshaped)
X_test_scaled = scaler.transform(X_test_reshaped)

X_train_scaled = X_train_scaled.reshape(
    n_samples,
    n_steps,
    n_features
)
X_val_scaled = X_val_scaled.reshape(
    n_val,
    n_steps,
    n_features)

X_test_scaled = X_test_scaled.reshape(
    n_test,
    n_steps,
    n_features)

y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).flatten()
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1)).flatten()

In [6]:
inputs = tf.keras.Input(shape=(X_train_scaled.shape[1], X_train_scaled.shape[2]))

x = layers.Bidirectional(layers.LSTM(64, return_sequences=False))(inputs)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation="relu")(x)
x = layers.Dropout(0.1)(x)
delta = layers.Dense(1)(x)  # predict deviation from mean

# Add the training mean as a constant offset
mean_rms = tf.constant(float(y_train.mean()), dtype=tf.float32)
outputs = delta + mean_rms  # model only needs to learn the residual

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    #loss=tf.keras.losses.Huber(delta=0.1),# lower delta = less mean-regression pull
    loss = 'mae',
    metrics=["mae"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)
]

history = model.fit(
    X_train_scaled, y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=50,          # let early stopping decide — 20 is too few
    batch_size=32,       # smaller batch = noisier gradients = better spike learning
    callbacks=callbacks,
    verbose=1)

Epoch 1/50


KeyboardInterrupt: 

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import r2_score
y_pred = model.predict(X_test_scaled).flatten()
y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
x_axis = np.arange(len(y_test))
plt.figure(figsize=(10,4))
plt.plot(x_axis[:200], y_test[:200], label="True")
plt.plot(x_axis[:200], y_pred[:200], label="Predicted")
plt.xlabel('Sample Index')
plt.ylabel('RMS Vertical Acceleration')
plt.title(f'LSTM model ($R^2$ = {r2_score(y_test, y_pred):.4f})')
plt.savefig('lstm_predictions_full.png')
plt.legend()
plt.show()

In [ ]:
plt.plot(y_test, y_pred, 'o')